# 02 — Fine-tune DistilBERT on Colab GPU

Runs `python -m src.models.train` (Phase 3, see `Architecture.md` / `Phases.md`) — the actual fine-tuning step, which needs a GPU this project's local host doesn't have.

Before running: **Runtime → Change runtime type → T4 GPU** (or better).

Inputs: `data/processed/{train,val,test}.parquet` (produced locally by `python -m src.data.preprocess`, Phase 1 — not regenerated here). Outputs: fine-tuned model → `models/distilbert-finetuned/`, metrics → `metrics/distilbert.json`.

Both `data/` and `models/` are git-ignored, so this notebook moves data in and results out via Google Drive rather than git.

In [ ]:
!nvidia-smi

## 1. Clone the repo and install dependencies

In [ ]:
import os

REPO_URL = "https://github.com/AniketPatil1710/real-time-content-moderation.git"
REPO_DIR = "/content/real-time-content-moderation"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
!pip install -q -r requirements.txt

## 2. Mount Drive and pull in the processed data

Set `DRIVE_DATA_DIR` to wherever you've uploaded `train.parquet` / `val.parquet` / `test.parquet` (~61MB total) in your Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_RUN_DIR = "/content/drive/MyDrive/modguard/runs"  # <-- checkpoints + final model land here

In [ ]:
import shutil
import subprocess
from pathlib import Path

required = ["train.parquet", "val.parquet", "test.parquet"]

# Search everywhere Drive might expose the files: My Drive, shared drives, and
# "shortcut" folders added from someone else's share link all mount under /content/drive/.
found = subprocess.run(
    ["find", "/content/drive", "-name", "train.parquet"], capture_output=True, text=True
).stdout.strip().splitlines()

if not found:
    raise FileNotFoundError(
        "No train.parquet found anywhere under /content/drive. "
        "Make sure the shared folder has been added to your Drive "
        "(right-click it in Drive > 'Organize' > 'Add shortcut to Drive'), then re-run this cell."
    )
if len(found) > 1:
    print(f"Multiple train.parquet found, using the first:\n" + "\n".join(found))

src_dir = Path(found[0]).parent
print(f"Using DRIVE_DATA_DIR = {src_dir}")

missing = [f for f in required if not (src_dir / f).exists()]
if missing:
    raise FileNotFoundError(f"{src_dir} is missing {missing} — upload the other split(s) alongside train.parquet.")

Path("data/processed").mkdir(parents=True, exist_ok=True)
for f in required:
    shutil.copy(src_dir / f, f"data/processed/{f}")

!ls -lh data/processed/

## 3. Train

Hyperparameters come from `configs/training.yaml` (not hardcoded here — edit that file, not this cell, to change them).

`--checkpoint-dir` points at Drive so per-epoch checkpoints (`save_strategy="epoch"`, kept via `save_total_limit=2`) survive a Colab disconnect.

**Caveat:** `run_train()` in `src/models/train.py` calls `trainer.train()` with no `resume_from_checkpoint` — so if the runtime disconnects mid-run, re-running this cell restarts training from epoch 0, it does not pick back up from the last saved checkpoint on Drive. Budget for one uninterrupted session (T4, batch 32, 3 epochs, ≤300k rows per `training.yaml` — expect on the order of an hour).

In [ ]:
!python -m src.models.train --checkpoint-dir {DRIVE_RUN_DIR}/distilbert-finetuned --metrics-path metrics/distilbert.json

## 4. Check results and copy metrics back

`metrics/distilbert.json` isn't git-ignored, so it can be committed straight from your local machine once you copy it there (or just eyeball it here).

In [ ]:
import json

with open("metrics/distilbert.json") as f:
    print(json.dumps(json.load(f), indent=2))

!cp metrics/distilbert.json {DRIVE_RUN_DIR}/distilbert-metrics.json

## 5. Phase 4 — Evaluation & threshold selection

Runs `src/evaluation/thresholds.py` against the fine-tuned model saved on Drive in step 3: per-label PR curves, a `block_threshold` (precision >= 0.90) and `flag_threshold` (precision >= 0.5) per label for the allow/flag/block decision logic, and an aggregated worst-50-false-positives/negatives error report. See `Memory.md` Decision #10 for why those two design choices were made this way.

Uses the same `DRIVE_RUN_DIR` from step 2 — no new variables to set.

In [ ]:
!python -m src.evaluation.thresholds --model-dir {DRIVE_RUN_DIR}/distilbert-finetuned

## 6. Check thresholds and copy Phase 4 outputs back to Drive

`configs/thresholds.json` and `metrics/error_analysis.md` aren't git-ignored, so they can be committed straight from your local machine once copied there. `metrics/pr_curves/*.png` isn't git-ignored either, but is copied to Drive here too for convenience.

In [ ]:
import json

with open("configs/thresholds.json") as f:
    print(json.dumps(json.load(f), indent=2))

!mkdir -p {DRIVE_RUN_DIR}/phase4
!cp configs/thresholds.json {DRIVE_RUN_DIR}/phase4/thresholds.json
!cp metrics/error_analysis.md {DRIVE_RUN_DIR}/phase4/error_analysis.md
!cp -r metrics/pr_curves {DRIVE_RUN_DIR}/phase4/pr_curves